> 在线讲义：[第 4 章 · Autograd：账本怎么记、怎么倒着算](https://wikieden.github.io/pytorch-inside-out/lessons/04-autograd.html) · 本书《PyTorch 从里到外》

# 第 4 章 · Autograd：账本怎么记、怎么倒着算

本章回答：`loss.backward()` 到底做了什么？账本（计算图）长什么样？为什么每步训练都要 `zero_grad`？原地修改为什么会报错？自己定义反向公式怎么写？

配套讲义：仓库里 `pytorch-deep-dive/notes/lessons/04-autograd.md`（讲解、动画、练习都在讲义里）。对应原文：00-overview.md §2.3；官方文档 Autograd mechanics（〔在线〕docs.pytorch.org/docs/stable/notes/autograd.html）；Karpathy micrograd 视频（〔在线〕）；本章数字都在本机 torch 2.15.0 上跑过（代码见本章末尾"代码对照"）

**怎么用**：在 `pytorch-deep-dive/solutions/chapter-code/` 下用 `uv run jupyter lab` 打开（或任何装了 torch 的环境）。从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。Mac 上能跑：计时类的格子在 Mac 上用 MPS，分布式的格子在本机起 2 个 Gloo 进程。

In [ ]:
import torch

torch.set_printoptions(precision=4, sci_mode=False)
print("准备好了", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 数值斜率 | (f(w+h) − f(w)) / h | `numeric_slope()` | ≈ 6.001 |
| 第二步 链式法则 | 2z·x、2z | `chain_example()` | z = 7、L = 49、(28, 14) |
| 第三步 账本 | Pow → Add → Mul / AccumulateGrad | `chain_example()` 的 names | 四个节点名 |
| 第四步 梯度累加 | 再反向一次 | 同上 `g2` | (56, 28) |
| 第五步 自定义 ReLU | 上游梯度 × [x > 0] | `myrelu_grad()` | [0, 1, 1]，gradcheck True |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `chain_example`**（来自 `ch04_autograd.py`）：z = w·x + b，L = z²；返回 (z, L, w.grad, b.grad, 账本节点名)。

In [ ]:
def chain_example(x=2.0, w=3.0, b=1.0):
    """z = w·x + b，L = z²；返回 (z, L, w.grad, b.grad, 账本节点名)。"""
    xt = torch.tensor(x)
    wt = torch.tensor(w, requires_grad=True)
    bt = torch.tensor(b, requires_grad=True)
    z = wt * xt + bt
    L = z**2
    names = [type(L.grad_fn).__name__, type(L.grad_fn.next_functions[0][0]).__name__]
    names += [type(f).__name__ for f, _ in L.grad_fn.next_functions[0][0].next_functions]
    L.backward(retain_graph=True)
    g1 = (wt.grad.item(), bt.grad.item())
    L.backward()
    g2 = (wt.grad.item(), bt.grad.item())
    return z.item(), L.item(), g1, g2, names

**函数 `inplace_error`**（来自 `ch04_autograd.py`）：下面要用到的小工具。

In [ ]:
def inplace_error() -> str:
    w = torch.tensor(3.0, requires_grad=True)
    a = w.exp()
    a.add_(1)
    try:
        a.sum().backward()
        return "没报错"
    except RuntimeError as e:
        return str(e).split(":")[0]

**类 `MyReLU`**（来自 `ch04_autograd.py`）：下面要用到的小工具。

In [ ]:
class MyReLU(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return x.clamp(min=0)

    @staticmethod
    def backward(ctx, grad_out):
        (x,) = ctx.saved_tensors
        return grad_out * (x > 0)

**函数 `myrelu_grad`**（来自 `ch04_autograd.py`）：下面要用到的小工具。

In [ ]:
def myrelu_grad():
    t = torch.tensor([-1.0, 2.0, 3.0], requires_grad=True)
    MyReLU.apply(t).sum().backward()
    ok = torch.autograd.gradcheck(MyReLU.apply, (torch.tensor([-1.0, 2.0, 3.0], dtype=torch.double, requires_grad=True),))
    return t.grad.tolist(), ok

**函数 `numeric_slope`**（来自 `ch04_autograd.py`）：L = w² 的数值导数：拧一点点看变化。

In [ ]:
def numeric_slope(w: float = 3.0, h: float = 1e-3) -> float:
    """L = w² 的数值导数：拧一点点看变化。"""
    return ((w + h) ** 2 - w**2) / h

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 L = w²，w = 3 的数值斜率 ≈", round(numeric_slope(), 3))
z, L, g1, g2, names = chain_example()

In [ ]:
print("\n第二步 z =", z, " L =", L, " (w.grad, b.grad) =", g1)

In [ ]:
print("\n第三步 账本：", names)

In [ ]:
print("\n第四步 再反向一次（不清零）：", g2)
print("原地修改：", inplace_error())

In [ ]:
print("\n第五步 MyReLU 梯度与 gradcheck：", myrelu_grad())

## 3. 代码里才看得到的

1. **`retain_graph=True` 才能反向两次**：默认反向完账本就释放了，第二次会报错。
2. **gradcheck 要 float64**：float32 的数值导数误差太大，会误报。

## 4. 自己改一改

- 把 `numeric_slope` 的 h 改成 1e-7，看 float 精度让结果变差。
- 给 `MyReLU` 故意写错反向（比如返回 `grad_out`），看 gradcheck 报什么。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 梯度是旋钮的敏感度，链式法则把每步倍数乘起来；
> 前向记节点，反向倒着走，结果存叶子；
> `.grad` 只会加，每步要清零；反向要用的数别原地改。